In [0]:
%run ./02_utility

In [0]:
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# silver.cash_transactions

In [0]:
log_checkpoint("silver_cash_transactions", "in_progress")

df_ct_bronze = spark.table(f"{catalog}.{bronze_schema}.cashtransaction") \
    .filter(col("_batch") <= batch_id)

df_ct_typed = df_ct_bronze.select(
    col("CT_CA_ID").cast("bigint").alias("CT_CA_ID"),
    col("CT_DTS").cast("timestamp").alias("CT_DTS"),
    col("CT_AMT").cast("decimal(12,2)").alias("CT_AMT"),
    col("CT_NAME").alias("CT_NAME"),
    col("_batch"),
    col("_ingest_ts")
)

In [0]:
display(df_ct_bronze)

In [0]:
dedup_window = Window.partitionBy("CT_CA_ID", "CT_DTS", "CT_AMT", "CT_NAME") \
                    .orderBy(col("_ingest_ts").desc())

df_ct_deduped = df_ct_typed.withColumn("_rn", row_number().over(dedup_window)) \
                           .filter(col("_rn") == 1) \
                            .drop("_rn", "_ingest_ts")



In [0]:
display(df_ct_deduped)

In [0]:
df_ct_silver = df_ct_deduped.withColumn("_load_ts", current_timestamp()) \
        .withColumn("_run_id", lit(run_id))

In [0]:
df_ct_silver.write.mode("overwrite") \
    .saveAsTable(f"{catalog}.{silver_schema}.cash_transactions")

In [0]:
ct_silver_count = spark.table(f"{catalog}.{silver_schema}.cash_transactions").count()
print(f"silver.cash_transactions rows: {ct_silver_count}")
log_audit("silver_cash_transactions", "create_or_replace", ct_silver_count)
log_checkpoint("silver_cash_transactions", "completed", ct_silver_count)

In [0]:
log_checkpoint("silver_account", "in_progress")

# silver.account is created by customer domain (silver_customer_customermgmt) in batch 1
# This notebook only merges Account.txt CDC for batches 2 and 3
if batch_id == "1":
    print("Batch 1: silver.account already created by customer domain. Skipping to validation.")

In [0]:
if batch_id in ("2", "3"):
    pass  # no XML validation needed - already in table from batch 1

In [0]:
df_batchdate = spark.table(f"{catalog}.{silver_schema}.batchdate").select(
    col("batchid").cast("string").alias("_batch"),
    col("batchdate").cast("timestamp").alias("batch_timestamp")
)

In [0]:
# Read Account.txt CDC only for batch 2/3
if batch_id in ("2", "3"):
    df_account_cdc = spark.table(f"{catalog}.{bronze_schema}.account") \
        .filter((col("CDC_FLAG") != "D") & (col("_batch") <= batch_id))

    df_account_cdc_ts = df_account_cdc.join(df_batchdate, on="_batch", how="left") \
        .select(
            col("CA_ID").cast("bigint").alias("CA_ID"),
            col("CA_C_ID").cast("bigint").alias("C_ID"),
            col("CA_B_ID").cast("bigint").alias("CA_B_ID"),
            col("CA_NAME").alias("CA_NAME"),
            col("CA_TAX_ST").cast("tinyint").alias("CA_TAX_ST"),
            col("CA_ST_ID").alias("Status"),
            col("batch_timestamp").alias("ActionTS"),
            col("_batch"),
        )

In [0]:
if batch_id in ("2", "3"):
    cdc_count = df_account_cdc_ts.count()
    print(f"Account.txt CDC (rows): {cdc_count}")
    if cdc_count > 0:
        print(f"CDC Flag breakdown:")
        spark.table(f"{catalog}.{bronze_schema}.account").filter(col("_batch") <= batch_id).groupby("CDC_FLAG").count().orderBy("CDC_FLAG").show()
else:
    print(f"No CDC for batch {batch_id}")

In [0]:
# For batch 2/3: use CDC data only (XML already in table from batch 1)
if batch_id in ("2", "3"):
    df_all_accounts = df_account_cdc_ts

In [0]:
if batch_id in ("2", "3"):
    dedup_window = Window.partitionBy("CA_ID").orderBy(col("ActionTS").desc())

    df_account_deduped = df_all_accounts.withColumn("_rn", row_number().over(dedup_window)) \
        .filter(col("_rn") == 1) \
        .drop("_rn")

In [0]:
if batch_id in ("2", "3"):
    df_account_silver = df_account_deduped.withColumn("_load_ts", current_timestamp()) \
        .withColumn("_run_id", lit(run_id))
    print(f"Account CDC records after deduplication: {df_account_silver.count()}")

In [0]:
if batch_id in ("2", "3"):
    display(df_account_silver)

In [0]:
target_table = f"{catalog}.{silver_schema}.account"

if batch_id in ("2", "3"):
    delta_target = DeltaTable.forName(spark, target_table)

    delta_target.alias("tgt") \
        .merge(df_account_silver.alias("src"), "tgt.CA_ID = src.CA_ID") \
        .whenMatchedUpdate(
            condition="src.ActionTS > tgt.ActionTS",
            set={
                "C_ID": "src.C_ID",
                "CA_B_ID": "src.CA_B_ID",
                "CA_NAME": "src.CA_NAME",
                "CA_TAX_ST": "src.CA_TAX_ST",
                "Status": "COALESCE(src.Status, tgt.Status)",
                "ActionTS": "src.ActionTS",
                "_batch": "src._batch",
                "_load_ts": "src._load_ts",
                "_run_id": "src._run_id",
            }
        ) \
        .whenNotMatchedInsert(
            values={
                "CA_ID": "src.CA_ID",
                "C_ID": "src.C_ID",
                "CA_B_ID": "src.CA_B_ID",
                "CA_NAME": "src.CA_NAME",
                "CA_TAX_ST": "src.CA_TAX_ST",
                "Status": "src.Status",
                "ActionTS": "src.ActionTS",
                "_batch": "src._batch",
                "_load_ts": "src._load_ts",
                "_run_id": "src._run_id",
            }
        ) \
        .execute()
    print("merge success")
else:
    print(f"Batch 1: silver.account exists (from customer domain)")

In [0]:
display(target_table)

In [0]:
account_silver_count = spark.table(target_table).count()
print(f"silver.account total rows: {account_silver_count}")

log_audit("silver.account", "merge" if batch_id in ("2", "3") else "skip_batch1", account_silver_count)
log_checkpoint("silver_account", "completed", account_silver_count)